# 12 · Baselines and ablation campaign
**Status: UNEXECUTED FIRST DRAFT.** No comparative result exists yet.

**Purpose:** train full AHR and three behavior-changing controls under identical observed data, optimizer-step targets, action budgets and seeds. **Inputs:** a selected supervised warm-start checkpoint, frozen replay, configuration and recovery gates. **Outputs:** a campaign manifest, independently resumable run checkpoints and measured run summaries. **Expected runtime:** one bounded training invocation per selected job; campaign duration depends on throughput and concurrency. **Checkpoint directory:** `checkpoints/<job.run_id>/`.

**Exact resume:** use the same campaign configuration, selected job and run ID. Each job checks data/source/config hashes and resumes its own optimizer state. A GPU count is a concurrency option, not a claim of distributed model sharding. Each full model must fit the assigned 96 GB card.

**Run All:** writes a reviewable eight-job plan (four variants × two seeds), then starts only the selected job if its prerequisites exist. The optional campaign launcher is defined but not automatically invoked. Its detached children load notebook definitions directly; no authored `.py` or shell helpers are created.

**Comparisons:** Euclidean event geometry; fixed Queen scheduling; Scout disabled. All preserve model tensor dimensions so the same supervised initialization can be used. These measure the effect of the corresponding component under the implemented training protocol; they are not a claim that AHR outperforms a separately optimized external agent.

## Load the complete training stack

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([11], globals())


## Declare paired variants and per-job GPUs

In [ ]:
import os
import subprocess
import sys


def make_ablation_plan(cfg, seeds=(42, 43), devices=None):
    devices = list(devices or [f'cuda:{i}' for i in range(8)])
    if not devices:
        raise ValueError('Provide at least one explicit device.')
    variants = {
        'full_ahr': {},
        'euclidean': {'geometry': 'euclidean'},
        'fixed_queen': {'queen_mode': 'fixed'},
        'no_scout': {'scout_enabled': False},
    }
    plan = []
    for seed in seeds:
        for name, change in variants.items():
            job_cfg = copy.deepcopy(cfg)
            job_cfg['reasoning'].update(change)
            job_cfg['train'].update({'stage': 'joint', 'seed': int(seed), 'device': devices[len(plan) % len(devices)]})
            plan.append({'variant': name, 'seed': int(seed), 'run_id': f'xl-ablation-{name}-seed{seed}-v1',
                         'device': job_cfg['train']['device'], 'config': job_cfg,
                         'matching': {'data': 'identical immutable replay', 'initialization': 'same supervised checkpoint',
                                      'updates': job_cfg['train']['max_steps'], 'action_budget': job_cfg['environment']['max_actions']}})
    return plan


def run_ablation_job(job, root=ROOT):
    problems = training_readiness(job['config'], 'joint', root)
    if problems:
        raise RuntimeError('Ablation prerequisites: ' + '; '.join(problems))
    return train_joint(job['config'], job['run_id'], root)


def launch_ablation_campaign(plan, max_parallel=2, root=ROOT):
    # Explicit optional launcher. Notebook-only source is loaded by a short Python expression.
    if not 1 <= int(max_parallel) <= 8:
        raise ValueError('max_parallel must be between one and eight.')
    devices = [job['device'] for job in plan]
    if len(devices) != len(set(devices)):
        raise ValueError('This simultaneous launcher requires unique devices per job; use run_ablation_job sequentially otherwise.')
    for job in plan:
        problems = training_readiness(job['config'], 'joint', root)
        if problems:
            raise RuntimeError(f'{job["run_id"]}: ' + '; '.join(problems))
    root = Path(root).resolve()
    campaign_path = root / 'config' / 'ablation_campaign.json'
    atomic_json(campaign_path, {'jobs': plan, 'source_hashes': source_manifest(root)})
    pending, running, completed = list(range(len(plan))), [], []
    try:
        while pending or running:
            while pending and len(running) < int(max_parallel):
                index = pending.pop(0)
                job = plan[index]
                log_path = root / 'logs' / f'{job["run_id"]}.log'
                log_path.parent.mkdir(parents=True, exist_ok=True)
                expression = (
                    "from pathlib import Path; import json; "
                    f"ROOT=Path({str(root)!r}); "
                    "nb=json.loads((ROOT/'notebooks'/'00_setup_and_hardware.ipynb').read_text()); "
                    "[exec(compile(''.join(c['source']),'00_setup_and_hardware.ipynb','exec'),globals()) for c in nb['cells'] if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])]; "
                    "load_exports([12],globals()); "
                    f"plan=json.loads(Path({str(campaign_path)!r}).read_text())['jobs']; "
                    f"run_ablation_job(plan[{index}],ROOT)"
                )
                with log_path.open('a') as log:
                    process = subprocess.Popen([sys.executable, '-u', '-c', expression], cwd=root, stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
                running.append((index, process, str(log_path)))
                print(f'Started {job["run_id"]} on {job["device"]}; PID {process.pid}; log {log_path}')
            for item in list(running):
                index, process, log_path = item
                status = process.poll()
                if status is not None:
                    completed.append({'run_id': plan[index]['run_id'], 'exit_code': status, 'log': log_path})
                    running.remove(item)
            if running:
                time.sleep(1)
    except KeyboardInterrupt:
        # SIGINT reaches the training boundary handler; no forced SIGKILL or deletion.
        import signal
        for _, process, _ in running:
            process.send_signal(signal.SIGINT)
        print('Sent interrupt to active jobs; allow checkpoint handlers to finish and inspect logs before restarting.')
        raise
    atomic_json(root / 'reports' / 'ablation_launch_results.json', completed)
    return completed


## Review the plan and execute one selected job

In [ ]:
cfg = load_config()
cfg['train']['stage'] = 'joint'
plan = make_ablation_plan(cfg)
atomic_json(ROOT / 'config' / 'ablation_campaign.json', {'jobs': plan, 'source_hashes': source_manifest(ROOT)})
for index, job in enumerate(plan):
    print(index, job['variant'], job['seed'], job['device'], job['run_id'])
SELECTED_JOB = 0
job = plan[SELECTED_JOB]
problems = training_readiness(job['config'], 'joint')
if problems:
    print('SELECTED ABLATION NOT STARTED:')
    for problem in problems:
        print(' •', problem)
else:
    ablation_report = run_ablation_job(job)
    print(ablation_report['status'])


## Optional parallel campaign
After reviewing memory usage, GPU anomalies, logs and the manifest, call `launch_ablation_campaign(plan, max_parallel=8)` in a new cell to use eight independent GPUs. This is independent-model training, not eight-way model sharding. Each child has its own lock and resumable run ID. The parent waits and records child exit codes; an exit code alone is not a benchmark result.

The wall-clock bound can leave variants at different optimizer steps. Continue each to the same target before making a training-budget-matched comparison. Notebook 13 also records action counts, inference time and model-call budgets; lower cost and higher completion are separate measurements.

## Inspect existing run reports without inventing results

In [ ]:
available_reports = []
for job in plan:
    path = ROOT / 'reports' / f'{job["run_id"]}.json'
    if path.exists():
        result = json.loads(path.read_text())
        available_reports.append({'variant': job['variant'], 'seed': job['seed'], 'status': result['status'],
                                  'step': result['completed_step_in_memory'], 'training_loss': result.get('metrics', {}).get('loss')})
print(available_reports if available_reports else 'No ablation training results yet.')
